# shor_15_final_compilation

This notebook performs the final Quantinuum-native resource estimate for the Steane-encoded Shor-15 circuit.

It intentionally starts from `full_steane_physical_k0_circuit`, the final circuit object produced by `Compilation/shor_15_steane_encoding.ipynb`. It does not rebuild the Shor or Steane construction pipeline here. If that circuit is not already in the current kernel, the notebook loads the QPY artifact saved by the Steane notebook.

The native compilation path is Qiskit `QuantumCircuit` -> TKET `Circuit` -> `QuantinuumBackend` compilation. The timing model below is a local native-gate lower bound: it excludes transport, cooling, queueing, Nexus overhead, and full hardware shot overhead.

In [1]:
import importlib
from pathlib import Path
from pprint import pprint
import sys

from IPython.display import Markdown, display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "Compilation" / "shor_compilation.py").exists():
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break
else:
    raise ImportError("Could not locate Compilation/shor_compilation.py")

from Compilation import shor_compilation
shor_compilation = importlib.reload(shor_compilation)

source_circuit = shor_compilation.require_full_steane_physical_k0_circuit(globals())
full_steane_physical_k0_circuit = source_circuit
source_summary = shor_compilation.source_circuit_summary(source_circuit)

display(Markdown("## Source circuit summary"))
pprint(source_summary)

## Source circuit summary

{'classical_bits': 117,
 'k0_parameters': {'N': 15, 'a': 11, 'n': 4},
 'logical_qubits': 10,
 'magic_refresh_counts': {'t': 24, 'tdg': 20},
 'maintenance_syndrome_rounds': 664,
 'name': 'first_pass_steane_physical_k0',
 'qubits': 117,
 'source_asap_layers': 86,
 'source_circuit': 'build_first_pass_reduced_10q_shor_15_gate_only_k0_layout',
 'top_level_operation_counts': {'barrier': 163,
                                'ch': 308,
                                'cx': 18550,
                                'h': 5196,
                                'if_else': 196,
                                'measure': 4881,
                                'reset': 7068,
                                'ry': 44,
                                's': 168,
                                'sdg': 168,
                                'x': 49}}


## Quantinuum/TKET dependencies

This notebook deliberately uses Quantinuum's TKET compiler path. If the TKET packages are missing, install them in the project environment:

```bash
pip install pytket pytket-qiskit pytket-quantinuum
```

In [2]:
tket_dependencies = shor_compilation.require_tket_quantinuum_dependencies()
print("TKET/Quantinuum dependencies are available.")

TKET/Quantinuum dependencies are available.


## Quantinuum native compilation

The compilation target is `H2-1E`, and the first pass uses `optimisation_level=0` to avoid aggressive optimization of the error-correction structure. This is a TKET/Quantinuum compilation step, not `qiskit.transpile`.

When the emulator name is not available through the local Quantinuum device list, the notebook uses the packaged `H2` target data from `pytket-quantinuum` for offline compilation only. This still calls `QuantinuumBackend.get_compiled_circuit`; it is not a manual rebase.

In [3]:
QUANTINUUM_DEVICE_NAME = shor_compilation.DEFAULT_QUANTINUUM_DEVICE_NAME
QUANTINUUM_OPTIMISATION_LEVEL = shor_compilation.DEFAULT_QUANTINUUM_OPTIMISATION_LEVEL

try:
    compilation_result = shor_compilation.compile_quantinuum_native(
        source_circuit,
        device_name=QUANTINUUM_DEVICE_NAME,
        optimisation_level=QUANTINUUM_OPTIMISATION_LEVEL,
    )
    compilation_error = None
except RuntimeError as exc:
    compilation_result = None
    compilation_error = exc

if compilation_result is None:
    tk_circuit = None
    compiled_tk_circuit = None
    compiled_qiskit_circuit = None
    compiled_summary = {
        "target": QUANTINUUM_DEVICE_NAME,
        "optimisation_level": QUANTINUUM_OPTIMISATION_LEVEL,
        "status": "blocked_before_quantinuum_compilation",
        "reason": str(compilation_error),
        "source_control_flow_like_counts": shor_compilation.source_control_flow_like_counts(source_circuit),
    }
else:
    tk_circuit = compilation_result["tk_circuit"]
    compiled_tk_circuit = compilation_result["compiled_tk_circuit"]
    compiled_qiskit_circuit = compilation_result["compiled_qiskit_circuit"]
    compiled_summary = compilation_result["compiled_summary"]

if compilation_result is not None and compilation_result["qiskit_conversion_error"] is not None:
    print(
        "TKET compilation succeeded, but conversion back to Qiskit failed: "
        f"{compilation_result['qiskit_conversion_error']}"
    )

display(Markdown("## Quantinuum-compiled circuit summary"))
pprint(compiled_summary)

H2-1E was not available from the local Quantinuum device list (DeviceNotAvailable: H2-1E). Using packaged H2 target data for offline compilation.


## Quantinuum-compiled circuit summary

{'backend_data_source': 'packaged pytket-quantinuum H2 data',
 'compiled_bits': 293,
 'compiled_qubits': 117,
 'compiled_tket_commands': 120399,
 'converted_back_to_qiskit': True,
 'optimisation_level': 0,
 'source_exceeds_target_nominal_qubits': True,
 'source_tket_commands': 36967,
 'target': 'H2-1E',
 'target_nominal_classical_registers': 4000,
 'target_nominal_qubits': 56}


## Native operation duration table

Public Quantinuum facts used here:

- H2 native TKET gate families include `OpType.PhasedX`, `OpType.Rz`, `OpType.ZZMax`, `OpType.ZZPhase`, and `OpType.TK2`; `Rz` is virtual in software.
- H2 product data gives 56 qubits and 4 parallel two-qubit operations for the current system model.
- Quantinuum's FAQ says exact transport timing for arbitrary H2 circuits is not exposed publicly, and that circuit timing depends heavily on transport and cooling.

The table below is therefore an explicit local gate-operation lower-bound model, not a full hardware shot-time model. Transport, cooling, queueing, Nexus overhead, and complete hardware shot overhead are excluded. Update the non-virtual durations if you have a course-provided or calibrated timing table.

In [4]:
duration_table = shor_compilation.duration_table()
shor_compilation.show_table(duration_table, index="operation_family")

,duration_us,note
operation_family,,
Rz,0.0,Virtual software-frame operation documented by...
PhasedX,10.0,Explicit lower-bound model input; replace with...
ZZMax,250.0,Explicit lower-bound model input; transport/co...
ZZPhase,250.0,Explicit lower-bound model input; transport/co...
Measure,400.0,Explicit lower-bound model input; full SPAM/sh...
Reset,400.0,Explicit lower-bound model input; full SPAM/sh...
Barrier,0.0,Scheduling/control metadata; zero-time bookkee...
Conditional,0.0,Scheduling/control metadata; zero-time bookkee...
SetBits,0.0,Scheduling/control metadata; zero-time bookkee...


In [5]:
if compiled_tk_circuit is None:
    analysis_result = None
    command_records = []
    display(Markdown("Native command analysis skipped because Quantinuum compilation did not complete."))
else:
    analysis_result = shor_compilation.analyze_compiled_tket_circuit(compiled_tk_circuit)
    command_records = analysis_result["command_records"]

## Native gate counts

In [6]:
if analysis_result is None:
    native_gate_counts = {}
    metadata_counts = {}
    raw_op_counts = {}
    native_gate_count_table = []
    metadata_count_table = []
else:
    native_gate_counts = analysis_result["native_gate_counts"]
    metadata_counts = analysis_result["metadata_counts"]
    raw_op_counts = analysis_result["raw_op_counts"]
    native_gate_count_table = analysis_result["native_gate_count_table"]
    metadata_count_table = analysis_result["metadata_count_table"]

display(Markdown("### Native gate counts"))
display(shor_compilation.show_table(native_gate_count_table, index="operation_family"))
display(Markdown("### Scheduling/control metadata counts"))
display(shor_compilation.show_table(metadata_count_table, index="operation_family"))

### Native gate counts

,count,duration_us,serial_time_us
operation_family,,,
Measure,4901,400.0,1960400.0
PhasedX,43505,10.0,435050.0
Reset,7228,400.0,2891200.0
Rz,45288,0.0,0.0
ZZPhase,19078,250.0,4769500.0


### Scheduling/control metadata counts

,count
operation_family,
Barrier,223
RangePredicate,176


## ASAP native moments

Commands are placed in the earliest moment that does not conflict with already scheduled commands on any shared qubit or classical bit. Barriers remain hard dependencies through their listed resources.

In [7]:
if analysis_result is None:
    scheduled_command_records = []
    moment_records = []
    total_time_us = None
else:
    scheduled_command_records = analysis_result["scheduled_command_records"]
    moment_records = analysis_result["moment_records"]
    total_time_us = analysis_result["total_time_us"]

display(Markdown("### First 25 native moments"))
display(shor_compilation.show_table(moment_records[:25], index="moment"))
display(Markdown("### Last 25 native moments"))
display(shor_compilation.show_table(moment_records[-25:], index="moment"))

### First 25 native moments

,duration_us,slowest_ops,command_count,operation_counts
moment,,,,
0,400.0,Reset,197,"{'Barrier': 11, 'RangePredicate': 176, 'Reset'..."
1,400.0,Reset,97,"{'Reset': 87, 'Rz': 10}"
2,10.0,PhasedX,21,"{'Barrier': 11, 'PhasedX': 10}"
3,10.0,PhasedX,81,"{'PhasedX': 1, 'Rz': 80}"
4,250.0,ZZPhase,78,"{'PhasedX': 76, 'Rz': 1, 'ZZPhase': 1}"
5,250.0,ZZPhase,36,"{'PhasedX': 1, 'Rz': 34, 'ZZPhase': 1}"
6,250.0,ZZPhase,35,"{'PhasedX': 1, 'Rz': 1, 'ZZPhase': 33}"
7,10.0,PhasedX,67,"{'PhasedX': 33, 'Rz': 34}"
8,250.0,ZZPhase,44,"{'PhasedX': 1, 'Rz': 32, 'ZZPhase': 11}"


### Last 25 native moments

,duration_us,slowest_ops,command_count,operation_counts
moment,,,,
3324,250.0,ZZPhase,5,"{'PhasedX': 2, 'Rz': 2, 'ZZPhase': 1}"
3325,400.0,Measure,5,"{'Measure': 1, 'PhasedX': 2, 'Rz': 1, 'ZZPhase..."
3326,400.0,Reset,4,"{'PhasedX': 1, 'Reset': 1, 'Rz': 1, 'ZZPhase': 1}"
3327,10.0,PhasedX,4,"{'PhasedX': 1, 'Rz': 3}"
3328,10.0,PhasedX,4,"{'PhasedX': 3, 'Rz': 1}"
3329,400.0,Measure,3,"{'Measure': 1, 'ZZPhase': 2}"
3330,400.0,Reset,4,"{'PhasedX': 2, 'Reset': 1, 'Rz': 1}"
3331,0.0,Rz,3,{'Rz': 3}
3332,10.0,PhasedX,3,{'PhasedX': 3}


## Final resource summary

In [8]:
if analysis_result is None:
    resource_summary = {
        "input_source": "full_steane_physical_k0_circuit from shor_15_steane_encoding.ipynb QPY artifact",
        "target": QUANTINUUM_DEVICE_NAME,
        "optimisation_level": QUANTINUUM_OPTIMISATION_LEVEL,
        "compilation_status": "blocked_before_quantinuum_compilation",
        "blocking_reason": str(compilation_error),
        "source_control_flow_like_counts": shor_compilation.source_control_flow_like_counts(source_circuit),
        "source_qubits": source_circuit.num_qubits,
        "source_classical_bits": source_circuit.num_clbits,
        "source_operation_counts": dict(source_circuit.count_ops()),
        "native_gate_counts": {},
        "metadata_counts": {},
        "raw_operation_counts": {},
        "native_moments": None,
        "total_gate_operation_time_us": None,
        "timing_scope": "Unavailable until Qiskit-to-TKET conversion succeeds.",
        "timing_model_sources": shor_compilation.TIMING_MODEL_SOURCE_URLS,
    }
else:
    resource_summary = shor_compilation.resource_summary(
        source_circuit,
        compiled_tk_circuit,
        analysis_result,
        compilation_result,
        device_name=QUANTINUUM_DEVICE_NAME,
        optimisation_level=QUANTINUUM_OPTIMISATION_LEVEL,
    )

pprint(resource_summary)

summary_table = [
    {"metric": key, "value": value}
    for key, value in resource_summary.items()
    if key not in {"native_gate_counts", "metadata_counts", "raw_operation_counts"}
]
shor_compilation.show_table(summary_table, index="metric")

{'backend_data_source': 'packaged pytket-quantinuum H2 data',
 'compiled_classical_bits': 293,
 'compiled_commands': 120399,
 'compiled_exceeds_target_nominal_qubits': True,
 'compiled_qubits': 117,
 'faithfulness_rule': 'Uses public a=11, N=15 arithmetic simplifications only; '
                      'does not use factors or a pre-known measured period.',
 'input_source': 'full_steane_physical_k0_circuit from '
                 'shor_15_steane_encoding.ipynb',
 'metadata_counts': {'Barrier': 223, 'RangePredicate': 176},
 'native_gate_counts': {'Measure': 4901,
                        'PhasedX': 43505,
                        'Reset': 7228,
                        'Rz': 45288,
                        'ZZPhase': 19078},
 'native_moments': 3349,
 'optimisation_level': 0,
 'raw_operation_counts': {'Barrier': 163,
                          'Conditional': 2692,
                          'Measure': 4881,
                          'PhasedX': 43005,
                          'RangePredicate': 1

,value
metric,
input_source,full_steane_physical_k0_circuit from shor_15_s...
target,H2-1E
optimisation_level,0
backend_data_source,packaged pytket-quantinuum H2 data
target_nominal_qubits,56
target_nominal_classical_registers,4000
compiled_exceeds_target_nominal_qubits,True
source_qubits,117
source_classical_bits,117
